# 🧹 Día 2 — Limpieza de Datos

**Proyecto:** Credit Risk Dashboard
**Dataset:** `credit_risk_dataset.csv` (32,581 filas × 12 columnas)
**Objetivo:** Aplicar las correcciones identificadas en el EDA (11 auditorías) y generar `credit_risk_clean.csv` listo para BigQuery.

## 📋 Plan de limpieza

| # | Acción | Filas afectadas |
|---|--------|-----------------|
| 1 | Eliminar duplicados exactos | 137 |
| 2 | Eliminar edades imposibles (>100) | 5 |
| 3 | Eliminar historial laboral imposible | 2 |
| 4 | Cap del historial crediticio inconsistente | 781 |
| 5 | Imputar `loan_int_rate` con la mediana | 3,116 |
| 6 | Imputar `person_emp_length` con 0 | 895 |
| 7 | Crear `n_solicitudes_cliente` | 32,581 |

**Resultado esperado:** ~32,437 filas × 13 columnas.

# BLOQUE A — SETUP Y CARGA

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo visual consistente con el Día 1
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Cargar el CSV crudo (el notebook está en notebooks/, el CSV en data/)
df = pd.read_csv("../data/credit_risk_dataset.csv")

# Guardar shape inicial como referencia para comparar al final
SHAPE_INICIAL = df.shape
print(f"✅ CSV crudo cargado.")
print(f"   Filas:    {SHAPE_INICIAL[0]:,}")
print(f"   Columnas: {SHAPE_INICIAL[1]}")
print(f"   Memoria:  {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

✅ CSV crudo cargado.
   Filas:    32,581
   Columnas: 12
   Memoria:  8.63 MB


## SNAPSHOT DEL ESTADO INICIAL (línea base)

In [2]:
print("=" * 60)
print("📊 ESTADO INICIAL DEL DATASET (antes de limpiar)")
print("=" * 60)

print(f"\n🔹 Shape:                    {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"🔹 Nulos totales:            {df.isnull().sum().sum():,}")
print(f"🔹 Duplicados exactos:       {df.duplicated().sum():,}")
print(f"🔹 Filas con algún nulo:     {df.isnull().any(axis=1).sum():,} "
      f"({df.isnull().any(axis=1).sum()/len(df)*100:.2f}%)")
print(f"🔹 Tasa de default global:   {df['loan_status'].mean()*100:.2f}%")
print(f"🔹 Edades > 100:             {(df['person_age'] > 100).sum()}")
print(f"🔹 Empleo > edad - 14:       {(df['person_emp_length'] > df['person_age'] - 14).sum()}")
print(f"🔹 Hist. cred. > edad - 18:  {(df['cb_person_cred_hist_length'] > df['person_age'] - 18).sum()}")

print("\n🔹 Nulos por columna (solo las que tienen):")
nulos = df.isnull().sum()
print(nulos[nulos > 0].to_string())

print("\n" + "=" * 60)

📊 ESTADO INICIAL DEL DATASET (antes de limpiar)

🔹 Shape:                    32,581 filas × 12 columnas
🔹 Nulos totales:            4,011
🔹 Duplicados exactos:       165
🔹 Filas con algún nulo:     3,943 (12.10%)
🔹 Tasa de default global:   21.82%
🔹 Edades > 100:             5
🔹 Empleo > edad - 14:       2
🔹 Hist. cred. > edad - 18:  781

🔹 Nulos por columna (solo las que tienen):
person_emp_length     895
loan_int_rate        3116



### ✅ Bloque A completado

- CSV crudo cargado: **32,581 filas × 12 columnas**.
- Línea base registrada para comparar al final del Día 2.
- Todos los problemas identificados en el EDA están presentes y cuantificados.

**Siguiente:** Bloque B — las 7 acciones de limpieza, una por una, cada una con su verificación antes/después.

### 🧹 Acción 1 — Eliminar duplicados exactos

**Problema detectado en el EDA:** 165 filas duplicadas exactamente (mismo patrón de valores y NaN).

**Estrategia:** `drop_duplicates()` conservando la primera ocurrencia. Los duplicados parciales (mismo perfil, distintos préstamos) se conservan.

**Criterio:** un duplicado exacto es un error de carga sin valor analítico.

In [3]:
filas_antes = len(df)

# Eliminar duplicados exactos conservando la primera ocurrencia
df = df.drop_duplicates(keep="first").reset_index(drop=True)

filas_despues = len(df)
filas_eliminadas = filas_antes - filas_despues

print(f"✅ Duplicados exactos eliminados")
print(f"   Filas antes:     {filas_antes:,}")
print(f"   Filas después:   {filas_despues:,}")
print(f"   Filas eliminadas: {filas_eliminadas:,} ({filas_eliminadas/filas_antes*100:.2f}%)")

✅ Duplicados exactos eliminados
   Filas antes:     32,581
   Filas después:   32,416
   Filas eliminadas: 165 (0.51%)


### 🧹 Acción 2 — Eliminar edades imposibles

**Problema:** 5 filas con `person_age` de 123 y 144 años (imposibles biológicamente).

**Estrategia:** eliminar filas con `person_age > 100`. Es una regla de negocio, no un umbral estadístico.

In [4]:
filas_antes = len(df)

# Mostrar los casos antes de eliminar (para documentar)
print("Filas con edad > 100 (a eliminar):")
print(df[df["person_age"] > 100][["person_age", "person_income", "loan_status"]].to_string())

# Eliminar
df = df[df["person_age"] <= 100].reset_index(drop=True)

filas_despues = len(df)
print(f"\n✅ Edades imposibles eliminadas")
print(f"   Filas eliminadas: {filas_antes - filas_despues}")
print(f"   Filas restantes:  {filas_despues:,}")

Filas con edad > 100 (a eliminar):
       person_age  person_income  loan_status
81            144         250000            0
183           144         200000            0
575           123          80004            0
747           123          78000            0
32132         144        6000000            0

✅ Edades imposibles eliminadas
   Filas eliminadas: 5
   Filas restantes:  32,411


### 🧹 Acción 3 — Eliminar historial laboral imposible

**Problema:** 2 filas con `person_emp_length = 123` en personas de 21-22 años (trabajarían desde antes de nacer).

**Estrategia:** eliminar filas donde `person_emp_length > person_age - 14` (asumiendo edad mínima laboral = 14 años).

In [5]:
filas_antes = len(df)

# Doble condición: NO nulo Y que viole la regla
mask_imposible = (
    df["person_emp_length"].notna() &
    (df["person_emp_length"] > (df["person_age"] - 14))
)

print("Filas con historial laboral imposible (a eliminar):")
print(df[mask_imposible][["person_age", "person_emp_length"]].to_string())

# Eliminar solo esos casos
df = df[~mask_imposible].reset_index(drop=True)

filas_despues = len(df)
print(f"\n✅ Historial laboral imposible eliminado")
print(f"   Filas eliminadas: {filas_antes - filas_despues}")
print(f"   Filas restantes:  {filas_despues:,}")

Filas con historial laboral imposible (a eliminar):
     person_age  person_emp_length
0            22              123.0
208          21              123.0

✅ Historial laboral imposible eliminado
   Filas eliminadas: 2
   Filas restantes:  32,409


### 🧹 Acción 4 — Cap del historial crediticio inconsistente

**Problema:** 781 filas con `cb_person_cred_hist_length > person_age - 18` (imposible haber construido ese historial).

**Estrategia:** en vez de eliminar (perderíamos 2.4% del dataset), **capear** al máximo plausible: `min(actual, person_age - 18)`.

**Justificación:** corrige la inconsistencia sin perder datos. Las 781 filas siguen siendo útiles para el análisis.

In [6]:
filas_antes = len(df)

# Identificar casos inconsistentes
mask_inconsistente = df["cb_person_cred_hist_length"] > (df["person_age"] - 18)
n_casos = mask_inconsistente.sum()

print(f"Casos detectados: {n_casos:,}")

# Mostrar algunos ejemplos antes del cap
print("\nEjemplos antes del cap:")
print(df[mask_inconsistente][["person_age", "cb_person_cred_hist_length"]].head(5).to_string())

# Aplicar cap: cb_person_cred_hist_length = min(actual, person_age - 18)
df.loc[mask_inconsistente, "cb_person_cred_hist_length"] = (
    df.loc[mask_inconsistente, "person_age"] - 18
)

# Verificar que ya no hay casos inconsistentes
casos_restantes = (df["cb_person_cred_hist_length"] > (df["person_age"] - 18)).sum()

print(f"\n✅ Cap aplicado")
print(f"   Casos corregidos:    {n_casos:,}")
print(f"   Casos restantes:     {casos_restantes}")
print(f"   Filas después:       {len(df):,} (sin cambios, solo modificación)")

Casos detectados: 773

Ejemplos antes del cap:
     person_age  cb_person_cred_hist_length
34           21                           4
59           21                           4
76           21                           4
146          21                           4
165          21                           4

✅ Cap aplicado
   Casos corregidos:    773
   Casos restantes:     0
   Filas después:       32,409 (sin cambios, solo modificación)


### 🧹 Acción 5 — Imputar `loan_int_rate` con la mediana

**Problema:** 3,116 nulos (9.56%). Verificado como MCAR en el EDA.

**Estrategia:** imputar con la mediana global.

**Justificación:** al ser MCAR, la mediana no sesga la distribución. Es robusta ante asimetría. No requiere imputación por grupo.

In [7]:
# ============================================
# ACCIÓN 5 — IMPUTAR loan_int_rate CON LA MEDIANA
# ============================================

nulos_antes = df["loan_int_rate"].isnull().sum()
mediana = df["loan_int_rate"].median()

print(f"Nulos antes:      {nulos_antes:,}")
print(f"Mediana calculada: {mediana:.4f}")

# Imputar
df["loan_int_rate"] = df["loan_int_rate"].fillna(mediana)

nulos_despues = df["loan_int_rate"].isnull().sum()

print(f"Nulos después:    {nulos_despues}")
print(f"\n✅ Imputación completada")

# Verificar que la distribución no se distorsionó
print(f"\nAntes (sin nulos):  media={df['loan_int_rate'].mean():.4f}, "
      f"mediana={df['loan_int_rate'].median():.4f}")
print(f"Filas: {len(df):,} (sin cambios)")

Nulos antes:      3,094
Mediana calculada: 10.9900
Nulos después:    0

✅ Imputación completada

Antes (sin nulos):  media=11.0145, mediana=10.9900
Filas: 32,409 (sin cambios)


### 🧹 Acción 6 — Imputar `person_emp_length` con 0

**Problema:** 895 nulos (2.75%).

**Estrategia:** imputar con 0, asumiendo que el nulo significa "desempleado" o "sin historial laboral".

**Justificación:** es una imputación con significado de negocio, no arbitraria. Alternativa (mediana) asignaría años de empleo a alguien que quizás nunca trabajó.

In [8]:
nulos_antes = df["person_emp_length"].isnull().sum()

print(f"Nulos antes: {nulos_antes:,}")

# Imputar con 0 (desempleado)
df["person_emp_length"] = df["person_emp_length"].fillna(0)

# Convertir a int64 (ya no hay nulos, podemos prescindir del float)
df["person_emp_length"] = df["person_emp_length"].astype(int)

nulos_despues = df["person_emp_length"].isnull().sum()
print(f"Nulos después: {nulos_despues}")
print(f"Tipo de dato:  {df['person_emp_length'].dtype}")

print(f"\n✅ Imputación completada")
print(f"Filas: {len(df):,} (sin cambios)")

Nulos antes: 887
Nulos después: 0
Tipo de dato:  int64

✅ Imputación completada
Filas: 32,409 (sin cambios)


### 🧹 Acción 7 — Crear la columna `n_solicitudes_cliente`

**Contexto:** en el EDA detectamos que 6,128 filas (18.81%) comparten el mismo perfil de cliente, con 2,531 clientes que tienen múltiples solicitudes.

**Estrategia:** crear una columna que cuente cuántas filas comparten el mismo perfil. Las columnas que definen al cliente son:
`person_age`, `person_income`, `person_home_ownership`, `person_emp_length`, `cb_person_default_on_file`, `cb_person_cred_hist_length`.

**Uso en el dashboard:** segmentar por "cliente de 1 solicitud" vs "cliente recurrente (2+ solicitudes)", un eje de riesgo adicional.

**Nota:** el conteo puede diferir ligeramente del EDA (2,531) porque:
1. Eliminamos 172 filas duplicadas/imposibles.
2. La imputación de `person_emp_length` con 0 agrupa a más perfiles.

In [9]:
# Columnas que definen al cliente
cols_cliente = [
    "person_age", "person_income", "person_home_ownership",
    "person_emp_length", "cb_person_default_on_file", 
    "cb_person_cred_hist_length"
]

# Contar cuántas filas comparten el mismo perfil
df["n_solicitudes_cliente"] = (
    df.groupby(cols_cliente)["loan_status"]
      .transform("size")
)

# Distribución
print("Distribución de solicitudes por cliente:")
print(df["n_solicitudes_cliente"].value_counts().sort_index().to_string())

# Métricas de negocio
n_clientes_unicos = df["n_solicitudes_cliente"].eq(1).sum()
n_clientes_repetidos = df["n_solicitudes_cliente"].gt(1).sum()

print(f"\n📊 Resumen:")
print(f"   Filas con 1 sola solicitud:    {n_clientes_unicos:,}")
print(f"   Filas en clientes recurrentes: {n_clientes_repetidos:,}")
print(f"   Máximo de solicitudes:         {df['n_solicitudes_cliente'].max()}")

# Comparar tasa de default por nº de solicitudes
print(f"\n📊 Tasa de default según nº de solicitudes:")
tasa = df.groupby("n_solicitudes_cliente")["loan_status"].agg(["mean", "count"])
tasa["mean"] = (tasa["mean"] * 100).round(2)
tasa.columns = ["tasa_default_%", "n"]
print(tasa.to_string())

print(f"\n✅ Columna n_solicitudes_cliente creada")
print(f"   Filas: {len(df):,}")
print(f"   Columnas: {len(df.columns)}")

Distribución de solicitudes por cliente:
n_solicitudes_cliente
1     26264
2      3880
3      1143
4       528
5       305
6       120
7       105
8        16
9        36
12       12

📊 Resumen:
   Filas con 1 sola solicitud:    26,264
   Filas en clientes recurrentes: 6,145
   Máximo de solicitudes:         12

📊 Tasa de default según nº de solicitudes:
                       tasa_default_%      n
n_solicitudes_cliente                       
1                               22.75  26264
2                               18.63   3880
3                               20.30   1143
4                               11.93    528
5                               16.07    305
6                               19.17    120
7                               18.10    105
8                               18.75     16
9                                2.78     36
12                               0.00     12

✅ Columna n_solicitudes_cliente creada
   Filas: 32,409
   Columnas: 13


### ✅ Bloque B completado — Las 7 acciones aplicadas

| # | Acción | Filas afectadas |
|---|--------|-----------------|
| 1 | Eliminar duplicados exactos | 165 |
| 2 | Eliminar edades imposibles | 5 |
| 3 | Eliminar historial laboral imposible | 2 |
| 4 | Cap del historial crediticio | 773 (modificadas) |
| 5 | Imputar `loan_int_rate` con mediana | 3,094 (modificadas) |
| 6 | Imputar `person_emp_length` con 0 | 887 (modificadas) |
| 7 | Crear `n_solicitudes_cliente` | 32,409 (nueva columna) |

**Filas eliminadas en total:** 172 (0.53%)
**Filas finales:** 32,409
**Columnas finales:** 13 (12 originales + `n_solicitudes_cliente`)

# BLOQUE C — COMPARATIVA INICIAL VS FINAL

In [10]:
print("=" * 60)
print("📊 COMPARATIVA — ANTES vs DESPUÉS DE LA LIMPIEZA")
print("=" * 60)

print(f"\n🔹 Shape:")
print(f"   Antes:  {SHAPE_INICIAL[0]:,} filas × {SHAPE_INICIAL[1]} columnas")
print(f"   Después: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"   Cambio:  -{SHAPE_INICIAL[0] - df.shape[0]:,} filas, +{df.shape[1] - SHAPE_INICIAL[1]} columna")

print(f"\n🔹 Nulos:")
print(f"   Antes:  4,011 nulos totales")
print(f"   Después: {df.isnull().sum().sum()} nulos")

print(f"\n🔹 Duplicados exactos:")
print(f"   Antes:  165")
print(f"   Después: {df.duplicated().sum()}")

print(f"\n🔹 Tasa de default global:")
print(f"   (antes del cap, la referencia era 21.82%)")
print(f"   Después: {df['loan_status'].mean()*100:.2f}%")

print(f"\n🔹 Valores imposibles:")
print(f"   Edades > 100:            {(df['person_age'] > 100).sum()}")
print(f"   Empleo > edad - 14:      {(df['person_emp_length'] > df['person_age'] - 14).sum()}")
print(f"   Hist. cred. > edad - 18: {(df['cb_person_cred_hist_length'] > df['person_age'] - 18).sum()}")

print("\n" + "=" * 60)

📊 COMPARATIVA — ANTES vs DESPUÉS DE LA LIMPIEZA

🔹 Shape:
   Antes:  32,581 filas × 12 columnas
   Después: 32,409 filas × 13 columnas
   Cambio:  -172 filas, +1 columna

🔹 Nulos:
   Antes:  4,011 nulos totales
   Después: 0 nulos

🔹 Duplicados exactos:
   Antes:  165
   Después: 1

🔹 Tasa de default global:
   (antes del cap, la referencia era 21.82%)
   Después: 21.87%

🔹 Valores imposibles:
   Edades > 100:            0
   Empleo > edad - 14:      0
   Hist. cred. > edad - 18: 0



# INVESTIGACIÓN DEL DUPLICADO POST-LIMPIEZA

In [11]:
print("=== DUPLICADOS EXACTOS POST-LIMPIEZA ===\n")

# Mostrar las filas duplicadas
dups = df[df.duplicated(keep=False)]
print(f"Filas involucradas: {len(dups)}")
print("\nContenido completo:")
print(dups.to_string())

# Comparar con el CSV limpio guardado
print(f"\nDuplicados en df (memoria):  {df.duplicated().sum()}")

=== DUPLICADOS EXACTOS POST-LIMPIEZA ===

Filas involucradas: 2

Contenido completo:
       person_age  person_income person_home_ownership  person_emp_length loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length  n_solicitudes_cliente
20513          27          30000                  RENT                  5    PERSONAL          B       4975          10.37            0                 0.17                         N                           9                      4
27999          27          30000                  RENT                  5    PERSONAL          B       4975          10.37            0                 0.17                         N                           9                      4

Duplicados en df (memoria):  1


# ELIMINAR EL DUPLICADO RESIDUAL POST-LIMPIEZA

In [12]:
filas_antes = len(df)

df = df.drop_duplicates(keep="first").reset_index(drop=True)

filas_despues = len(df)

print(f"✅ Duplicado residual eliminado")
print(f"   Filas antes:   {filas_antes:,}")
print(f"   Filas después: {filas_despues:,}")
print(f"   Eliminadas:    {filas_antes - filas_despues}")
print(f"   Duplicados restantes: {df.duplicated().sum()}")

# Re-guardar el CSV limpio
df.to_csv("../data/credit_risk_clean.csv", index=False)
print(f"\n✅ CSV re-guardado con {len(df):,} filas")

✅ Duplicado residual eliminado
   Filas antes:   32,409
   Filas después: 32,408
   Eliminadas:    1
   Duplicados restantes: 0

✅ CSV re-guardado con 32,408 filas


### 🔄 Iteración post-limpieza

Después de aplicar las 7 acciones, detecté **1 duplicado residual** que se creó durante la limpieza. Este es un fenómeno conocido: el cap del historial crediticio hizo que dos filas con `cb_person_cred_hist_length` distinto (que se capearon al mismo valor) quedaran idénticas en todas las columnas.

Se eliminó con `drop_duplicates()` y se re-guardó el CSV. **Lección:** la limpieza es iterativa, no un proceso único. Cada transformación puede crear nuevos problemas que hay que verificar.

**Filas finales:** 32,408 (eliminadas en total: 173, equivalente al 0.53% del dataset original).

# 📊 Resumen Ejecutivo — Día 2: Limpieza de Datos

**Proyecto:** Credit Risk Dashboard
**Fecha:** Día 2 de 7
**Notebook:** `02_limpieza.ipynb`

---

## 🎯 Objetivo

Aplicar las 7 acciones de limpieza identificadas en el EDA sobre las 32,581 solicitudes de crédito, y generar un CSV limpio (`credit_risk_clean.csv`) listo para cargar a BigQuery.

---

## 📋 Acciones aplicadas

| # | Acción | Filas afectadas | Estrategia |
|---|--------|-----------------|------------|
| 1 | Eliminar duplicados exactos | 165 eliminadas | `drop_duplicates(keep="first")` |
| 2 | Eliminar edades imposibles (`age > 100`) | 5 eliminadas | Filtro de regla de negocio |
| 3 | Eliminar historial laboral imposible | 2 eliminadas | `notna() & (emp_length > age - 14)` |
| 4 | Cap del historial crediticio inconsistente | 773 corregidas | `min(actual, age - 18)` |
| 5 | Imputar `loan_int_rate` con mediana | 3,094 rellenadas | `fillna(median)` |
| 6 | Imputar `person_emp_length` con 0 | 887 rellenadas | `fillna(0)` + cast a `int64` |
| 7 | Crear `n_solicitudes_cliente` | 32,408 nuevas | `groupby().transform("size")` |

---

## 📊 Resultado final

| Métrica | Antes | Después | Cambio |
|---------|-------|---------|--------|
| Filas | 32,581 | 32,408 | **-173 (0.53%)** |
| Columnas | 12 | 13 | +1 |
| Nulos totales | 4,011 | 0 | **-100%** |
| Duplicados exactos | 165 | 0 | **-100%** |
| Valores imposibles | 788 | 0 | **-100%** |
| Tasa de default global | 21.82% | 21.87% | +0.05 pp |

**Conclusión:** el dataset preserva el **99.47% de los datos originales**. Solo se eliminaron 173 filas, y todas las correcciones fueron quirúrgicas.

---

## 🏆 Hallazgos estrella del Día 2

### 🥇 Insight nuevo: la tasa de default BAJA con más solicitudes

| Nº solicitudes | Tasa de default | n |
|----------------|-----------------|---|
| 1 | 22.75% | 26,264 |
| 2 | 18.63% | 3,880 |
| 3 | 20.30% | 1,143 |
| 4 | **11.93%** | 528 |
| 5 | 16.07% | 305 |
| ... | ... | ... |
| 9 | **2.78%** | 36 |
| 12 | **0.00%** | 12 |

**Interpretación:** confirma el sesgo detectado en el EDA. El banco "retiene" a los clientes que pagan bien — los clientes con default son one-shot. Esto convierte a `n_solicitudes_cliente` en una **variable predictiva potente**, no solo informativa.

### 🥈 Error detectado y corregido: el NaN silencioso

Durante la Acción 3 (eliminar historial laboral imposible), el filtro `df[df["person_emp_length"] <= df["person_age"] - 14]` eliminó **889 filas en lugar de 2**. Causa: pandas evalúa `NaN <= algo` como `False`, así que todos los nulos de `person_emp_length` (895 filas) fueron descartados también.

**Corrección:** doble condición con `notna()`:
```python
mask_imposible = (
    df["person_emp_length"].notna() &
    (df["person_emp_length"] > (df["person_age"] - 14))
)